# Getting started with QuantLab

This notebook exercises the laboratory using **fictional deterministic prices**. No performance shown here is market evidence. Core logic lives in `src/quantlab`. Run all cells after installing the package. The default interval ends before the declared test period.

For real history, download through the CLI and replace the snapshot load; use the real reference YAML. Review RESEARCH_READINESS.md first.

In [ ]:
from pathlib import Path
import os
if Path.cwd().name == 'notebooks':
    os.chdir('..')
import quantlab
from quantlab.data.synthetic import synthetic
from quantlab.data.core import Dataset
from quantlab.experiments.config import ExperimentConfig
from quantlab.experiments.runner import run_experiment, Experiment
from quantlab.experiments.registry import ExperimentRegistry
from quantlab.analytics.plots import equity_plot, drawdown_plot, rolling_plot
from quantlab.analytics.metrics import subperiods
from quantlab.analytics.robustness import cost_sensitivity, parameter_sensitivity

## Inspect the data and provenance
The UP series includes a controlled two-for-one split. Flat closes are deliberately flagged. Missing sessions are fatal; no forward filling occurs.

In [ ]:
snapshot = synthetic().save('data')
dataset = Dataset.load(snapshot)
display(dataset.frame.head(), dataset.metadata, dataset.quality)

## Explicit experiment configuration
252-session endpoint, skip 21 sessions; monthly equal-weight selection. The fictional example holds at most two securities with a 25% per-position cap, leaving cash. This is a visible cash/exposure example, not an optimized choice.

In [ ]:
config = ExperimentConfig.load('configs/momentum_synthetic.yaml')
config.dataset = str(snapshot)
display(config.model_dump(), config.universe.model_dump())

## Run, persist, and inspect
Every completed run receives an ID; each repeated notebook execution creates new experiments rather than overwriting old evidence.

In [ ]:
experiment = run_experiment(config)
display(experiment.path, experiment.metrics)

In [ ]:
equity_plot(experiment).show()

In [ ]:
drawdown_plot(experiment).show()

In [ ]:
rolling_plot(experiment).show()

## Inspect execution and holdings
Compare signal_date to date. Check cash and exposure alongside returns; control outperformance is not an alpha claim.

In [ ]:
display(experiment.tables['trades'].head(12), experiment.tables['holdings'].tail(8), experiment.tables['returns'].tail())

## Predeclared cost sensitivity
All five assumptions (0, 5, 10, 25, 50 bps total one-way cost) are persisted. No best result is selected. This excludes the test interval.

In [ ]:
cost_results = cost_sensitivity(experiment)
display(cost_results)

## Predeclared parameter sensitivity
126, 189 and 252 session endpoints; skip stays fixed. These are comparisons to reveal fragility, not a search for a winner.

In [ ]:
parameter_results = parameter_sensitivity(experiment)
display(parameter_results)

## Subperiod analysis and experiment history
The untouched test is absent. Annualization on short periods is unstable. Repeated research creates multiple-testing risk even if every run is retained.

In [ ]:
display(subperiods(experiment.tables['returns']), ExperimentRegistry().list()[['id', 'purpose', 'parent_id', 'sharpe', 'cagr']])

## Reload and reproduce
Reload verifies immutable artifact hashes; reproduction compares every result table exactly. Preserve the dataset snapshot and environment. The report embeds interactive Plotly and opens without a server.

In [ ]:
loaded = Experiment.load(experiment.path.name)
reproduced = loaded.reproduce()
from quantlab.analytics.reporting import report
display(report(loaded), reproduced.path)

## Next research step
Write a hypothesis and predeclare its universe, periods and assumptions before editing a strategy. Add a `targets(history)` implementation in a normal Python module. Never tune against a consumed holdout. See DECISIONS_REQUIRED.md for decisions this platform deliberately leaves to you.